In [ ]:
# Persona API Image Extractor
# Extracts all image URLs from Persona inquiries
import sys, platform
print(sys.version)
print(sys.executable)


In [ ]:
# (API key is loaded in the main processing cell below)


In [ ]:
# ============================================================================
# MAIN PROCESSING: Extract image URLs from Persona inquiries
# ============================================================================
# Fetches inquiry data and extracts all image/photo URLs.
# Output saved to CSV with inquiry_id and list of image URLs.
# ============================================================================

import os
import csv
import time
import json
from typing import Dict, Any, List, Optional

import requests
from dotenv import load_dotenv

# ===============================
# Config
# ===============================
REQUEST_TIMEOUT = 30
SLEEP_S = 0.25  # be kind to API

from pathlib import Path
load_dotenv()
PERSONA_API_KEY = os.getenv("PERSONA_API_KEY")
if not PERSONA_API_KEY:
    raise RuntimeError("Missing PERSONA_API_KEY in .env")

# ===============================
# Persona API
# ===============================
def persona_get_inquiry(inquiry_id: str, include_verifications: bool = True) -> Dict[str, Any]:
    """Retrieve an inquiry.
    
    According to Persona API docs, verifications are included by default,
    but we can explicitly request them with the include parameter.
    """
    from urllib.parse import quote
    encoded_id = quote(inquiry_id, safe='')
    
    url = f"https://api.withpersona.com/api/v1/inquiries/{encoded_id}"
    headers = {
        "Authorization": f"Bearer {PERSONA_API_KEY}",
        "Accept": "application/json",
    }
    
    params = {}
    if include_verifications:
        # Explicitly include verifications to ensure we get full details
        params["include"] = "verifications"
    
    r = requests.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT)
    if r.status_code == 404:
        raise ValueError(f"Inquiry {inquiry_id} not found (404).")
    r.raise_for_status()
    return r.json()

def persona_get_verification(verification_id: str) -> Dict[str, Any]:
    """Retrieve a verification directly from the verification API endpoint.
    
    According to Persona API docs: https://docs.withpersona.com/2023-01-05/api-reference/verifications/retrieve-a-verification
    This endpoint returns full verification details including attributes.
    """
    from urllib.parse import quote
    encoded_id = quote(verification_id, safe='')
    
    url = f"https://api.withpersona.com/api/v1/verifications/{encoded_id}"
    headers = {
        "Authorization": f"Bearer {PERSONA_API_KEY}",
        "Accept": "application/json",
    }
    
    r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    if r.status_code == 404:
        raise ValueError(f"Verification {verification_id} not found (404).")
    r.raise_for_status()
    return r.json()

def get_last_government_id_verification(payload: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Get the last (most recent) verification/government-id from inquiry payload."""
    verifications = []
    included = payload.get("included", [])
    
    for item in included:
        if isinstance(item, dict):
            item_type = item.get("type", "")
            if item_type == "verification/government-id":
                verifications.append(item)
    
    # Return the last verification (most recent)
    return verifications[-1] if verifications else None

def extract_front_photo_url_from_last_verification(payload: Dict[str, Any]) -> Optional[str]:
    """Extract front-photo-url from the last (most recent) government-id verification."""
    verification = get_last_government_id_verification(payload)
    
    if verification:
        attributes = verification.get("attributes", {})
        front_photo_url = attributes.get("front-photo-url")
        if isinstance(front_photo_url, str) and front_photo_url:
            return front_photo_url
    
    return None

# ===============================
# Main CSV pipeline
# ===============================
def extract_front_photos_from_csv(input_csv: str, output_csv: str):
    """Read inquiry IDs from CSV and extract front-photo-url from the last verification."""
    fieldnames = ["file_name", "inquiry_id", "front_photo_url", "status"]
    
    # Read input CSV
    with open(input_csv, newline="", encoding="utf-8") as fin:
        reader = csv.DictReader(fin)
        
        # Get available column names (case-insensitive matching)
        if not reader.fieldnames:
            raise ValueError(f"CSV file {input_csv} appears to be empty or invalid.")
        
        # Normalize column names for case-insensitive matching
        # Create mapping: lowercase -> original name
        fieldnames_lower = {name.lower().strip(): name for name in reader.fieldnames}
        
        # Try to find inquiry_id and file_name columns
        inquiry_id_key = None
        file_name_key = None
        
        # Common variations of column names
        inquiry_id_variants = ["inquiry_id", "inquiry id", "inquiryid", "id", "inquiry"]
        file_name_variants = ["file name", "filename", "file_name", "File name"]
        
        for variant in inquiry_id_variants:
            variant_lower = variant.lower().strip()
            if variant_lower in fieldnames_lower:
                inquiry_id_key = fieldnames_lower[variant_lower]
                break
        
        for variant in file_name_variants:
            variant_lower = variant.lower().strip()
            if variant_lower in fieldnames_lower:
                file_name_key = fieldnames_lower[variant_lower]
                break
        
        if not inquiry_id_key:
            print(f"Available columns: {', '.join(reader.fieldnames)}")
            raise ValueError(f"Could not find inquiry_id column. Looking for: {', '.join(inquiry_id_variants)}")
        
        # File name is optional
        if file_name_key:
            print(f"Using columns: '{inquiry_id_key}' for inquiry_id, '{file_name_key}' for file_name\n")
        else:
            print(f"Using column: '{inquiry_id_key}' for inquiry_id (file_name column not found, will use inquiry_id as name)\n")
        
        # Extract inquiries
        inquiries = []
        for row in reader:
            inquiry_id = row.get(inquiry_id_key, "").strip()
            file_name = row.get(file_name_key, "").strip() if file_name_key else inquiry_id
            
            if inquiry_id:
                inquiries.append({"inquiry_id": inquiry_id, "file_name": file_name or inquiry_id})
    
    if not inquiries:
        raise ValueError(f"No inquiries found in {input_csv}. Check that 'inquiry_id' column exists and has data.")
    
    print(f"Found {len(inquiries)} inquiries to process\n")
    
    # Process inquiries and save results
    with open(output_csv, "w", newline="", encoding="utf-8") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        writer.writeheader()
        
        for i, inquiry in enumerate(inquiries, start=1):
            inquiry_id = inquiry["inquiry_id"]
            file_name = inquiry["file_name"]
            
            out = {
                "file_name": file_name,
                "inquiry_id": inquiry_id,
                "front_photo_url": "",
                "status": "OK"
            }
            
            print(f"[{i}/{len(inquiries)}] {file_name} ({inquiry_id})... ", end="", flush=True)
            try:
                payload = persona_get_inquiry(inquiry_id)
                
                front_photo_url = extract_front_photo_url_from_last_verification(payload)
                
                if front_photo_url:
                    out["front_photo_url"] = front_photo_url
                    print(f"✅ Found front photo")
                else:
                    out["status"] = "No government-id verification or no front photo"
                    print(f"⚠️ No front photo found")
                    
            except ValueError as e:
                out["status"] = f"Not found: {e}"
                print(f"❌ {e}")
            except requests.exceptions.HTTPError as e:
                status = getattr(e.response, 'status_code', 'unknown')
                out["status"] = f"HTTP {status}: {str(e)[:100]}"
                print(f"❌ HTTP {status}")
            except Exception as e:
                out["status"] = f"Error: {type(e).__name__}: {str(e)[:100]}"
                print(f"❌ {type(e).__name__}: {str(e)[:100]}")
            
            writer.writerow(out)
            time.sleep(SLEEP_S)
    
    print(f"\n✅ Done! Front photo URLs saved to: {output_csv}")
    
    # Summary
    import pandas as pd
    summary_df = pd.read_csv(output_csv)
    total = len(summary_df)
    with_photos = len(summary_df[summary_df['front_photo_url'].notna() & (summary_df['front_photo_url'] != '')])
    without_photos = total - with_photos
    
    print(f"\n{'='*60}")
    print(f"SUMMARY")
    print(f"{'='*60}")
    print(f"Total inquiries processed: {total}")
    print(f"Inquiries with front photo: {with_photos} ({with_photos/total*100:.1f}%)")
    print(f"Inquiries without front photo: {without_photos} ({without_photos/total*100:.1f}%)")
    print(f"{'='*60}")

# Run it - all outputs go to the same folder so they update together
OUTPUT_DIR = Path.cwd() / "Persona" if (Path.cwd() / "Persona").is_dir() else Path.cwd()
INPUT_CSV_PATH = OUTPUT_DIR / "Persona inquiries.csv" if (OUTPUT_DIR / "Persona inquiries.csv").exists() else Path("Persona inquiries.csv")
OUTPUT_CSV_PATH = OUTPUT_DIR / "inquiry_front_photos.csv"
print(f"📁 Output directory: {OUTPUT_DIR.absolute()}")
print(f"   Input CSV:  {INPUT_CSV_PATH}")
print(f"   Output CSV: {OUTPUT_CSV_PATH}
")

try:
    extract_front_photos_from_csv(str(INPUT_CSV_PATH), str(OUTPUT_CSV_PATH))
except FileNotFoundError:
    print(f"❌ File not found: {INPUT_CSV_PATH}")
    print(f"\nPlease ensure the CSV file exists with an 'inquiry_id' column.")
except Exception as e:
    print(f"❌ Error: {e}")


In [ ]:
# ============================================================================
# BULK DOWNLOAD PHOTOS: Download all photos and create ZIP file
# ============================================================================
# Downloads all photos from inquiry_front_photos.csv and creates a ZIP archive
# ============================================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import requests
import pandas as pd
import zipfile
import tempfile
import shutil
import pandas as pd


def download_photos_to_zip(csv_file: str = "inquiry_front_photos.csv", zip_filename: str = "persona_photos.zip", temp_dir: str = None, output_photos_dir: str = None):
    """Download all photos from the results CSV, optionally save to a folder, and create a ZIP file."""
    
    # Create directory for downloads (persistent folder or temp)
    if temp_dir:
        temp_path = Path(temp_dir)
        temp_path.mkdir(parents=True, exist_ok=True)
        is_temp = False
    elif output_photos_dir:
        temp_path = Path(output_photos_dir)
        temp_path.mkdir(parents=True, exist_ok=True)
        # Clear folder so it only contains this run's photos
        for f in temp_path.iterdir():
            f.unlink()
        is_temp = False
    else:
        temp_path = Path(tempfile.mkdtemp(prefix="persona_photos_"))
        is_temp = True
    
    # Read CSV
    df = pd.read_csv(csv_file)
    
    # Filter to only rows with photos (handle NaN and empty strings)
    df_with_photos = df[df['front_photo_url'].notna() & (df['front_photo_url'] != '')]
    
    if len(df_with_photos) == 0:
        print("No photos to download.")
        return None
    
    print(f"Downloading {len(df_with_photos)} photos...\n")
    
    downloaded = 0
    failed = 0
    downloaded_files = []  # list of (path, zip_entry_name) for unique names in ZIP
    used_names = set()
    
    for idx, row in df_with_photos.iterrows():
        inquiry_id = str(row['inquiry_id'])
        file_name = str(row.get('file_name', inquiry_id)) if pd.notna(row.get('file_name')) else inquiry_id
        photo_url = str(row['front_photo_url'])
        
        # Clean file name for filesystem
        safe_file_name = "".join(c for c in file_name if c.isalnum() or c in (' ', '-', '_')).strip()
        if not safe_file_name:
            safe_file_name = inquiry_id
        
        # Determine file extension from URL or use .jpg as default
        parsed_url = urlparse(photo_url)
        path = parsed_url.path
        if path.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
            ext = Path(path).suffix
        else:
            ext = '.jpg'  # Default
        
        # Unique name for ZIP (avoid duplicate filenames)
        base = f"{safe_file_name}{ext}"
        zip_name = base
        if base in used_names:
            zip_name = f"{safe_file_name}_{inquiry_id}{ext}"
        used_names.add(zip_name)
        output_file = temp_path / zip_name
        
        print(f"[{downloaded + failed + 1}/{len(df_with_photos)}] {file_name}... ", end="", flush=True)
        
        try:
            response = requests.get(photo_url, timeout=30, stream=True)
            response.raise_for_status()
            with open(output_file, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            downloaded_files.append((output_file, zip_name))
            print(f"✅ Downloaded")
            downloaded += 1
        except Exception as e:
            print(f"❌ Failed: {str(e)[:50]}")
            failed += 1
    
    # Create ZIP file
    zip_path = Path(zip_filename)
    print(f"\n📦 Creating ZIP file: {zip_path.name}...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for photo_file, zip_name in downloaded_files:
            zipf.write(photo_file, zip_name)
    
    if is_temp:
        shutil.rmtree(temp_path)
        print(f"🧹 Cleaned up temporary directory")
    elif output_photos_dir:
        print(f"📁 Photos also saved to: {Path(output_photos_dir).absolute()}")
    
    print(f"\n✅ Done! Downloaded {downloaded} photos, {failed} failed")
    print(f"📦 ZIP file created: {zip_path.absolute()}")
    print(f"   File size: {zip_path.stat().st_size / (1024*1024):.2f} MB")
    return str(zip_path.absolute())

# Run it - use same OUTPUT_DIR as the main processing cell so CSV and ZIP stay in sync
OUTPUT_DIR = Path.cwd() / "Persona" if (Path.cwd() / "Persona").is_dir() else Path.cwd()
try:
    zip_file = download_photos_to_zip(
        csv_file=str(OUTPUT_DIR / "inquiry_front_photos.csv"),
        zip_filename=str(OUTPUT_DIR / "persona_photos.zip"),
        output_photos_dir=str(OUTPUT_DIR / "persona_photos"),  # folder updated each run
    )
    if zip_file:
        print(f"\n💡 To find the ZIP file:")
        print(f"   Open Finder and press Cmd+Shift+G")
        print(f"   Paste this path: {Path(zip_file).parent}")
        print(f"   Or run: open {Path(zip_file).parent}")
except FileNotFoundError:
    print("❌ Results file not found. Run Cell 1 first to process inquiries.")
except Exception as e:
    print(f"❌ Error: {e}")


In [ ]:
# how many users lacking names in our templates?
# get the images
